# MODTRAN LUT を使った水蒸気 H2O フィッティング


- MODTRAN で作った LUT CSV: `Waveln,0.00,0.25,...` の形式
- HISUI スペクトル CSV: `y,x,wave_...nm` の横持ち形式

In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import least_squares

## 1. 設定

In [ ]:
# MODTRAN LUT CSV. Format: first column is wavelength, remaining columns are H2O values.
lut_csv = Path(r"E:\research\CH4\5p00.csv")

# HISUI spectra CSV made by hisui_region_spectra_export.ipynb.
hisui_spectra_csv = Path(r"D:\research\data\hisui_region_spectra.csv")

# Pixel selection from the HISUI spectra CSV.
# If pixel_y and pixel_x are not None, the row matching y,x is used.
# If either is None, hisui_row_index is used instead.
pixel_y = 0
pixel_x = 0
hisui_row_index = 0

# H2O fitting band selection on the HISUI side.
# Use "index" to select by HISUI band index in the wide CSV after y,x.
# Use "wavelength" to select by wavelength range.
hisui_selection_mode = "index"
hisui_band_start = 69       # inclusive, full HISUI band index. Example: 69 ~= 1038.40 nm
hisui_band_stop = 86        # exclusive. Example: 86 includes through ~= 1238.24 nm
hisui_wavelength_min_nm = 1030.0
hisui_wavelength_max_nm = 1245.0

# LUT selection. "index" keeps the same style as the original code; "wavelength" is easier to read.
lut_selection_mode = "index"
lut_index_start = 645       # inclusive. Example: around 1025 nm if LUT has 1 nm spacing from 380 nm
lut_index_stop = 865        # exclusive. Example: around 1244 nm
lut_wavelength_min_nm = 1025.0
lut_wavelength_max_nm = 1245.0

lut_scale_factor = 100.0
hisui_scale_factor = 1.0

# Initial parameters for LM: [sigma, mu, a, b, h2o, k]
initial_sigma = 6.5
initial_mu = 0.0
initial_a = 0.0
initial_h2o = 1.8
initial_k = 0.0

# If True, b is estimated from the current pixel before LM starts, as in the original notebook.
estimate_initial_b_from_pixel = True
initial_b = 1.0

# LM settings
least_squares_method = "lm"
max_nfev = 2000

## 2. 読み込みと整形の関数

In [ ]:
def parse_lut_parameter(column_name):
    text = str(column_name).strip().replace("p", ".")
    try:
        return float(text)
    except ValueError as exc:
        raise ValueError(f"Could not parse LUT parameter column as a number: {column_name!r}") from exc


def load_lut_csv(path, scale_factor=1.0):
    """Load LUT as a matrix: row 0 wavelength, rows 1..N spectra for each H2O value."""
    df = pd.read_csv(path)
    if df.shape[1] < 2:
        raise ValueError("LUT CSV must have wavelength plus at least one H2O column.")

    wavelength_col = df.columns[0]
    wavelengths = pd.to_numeric(df[wavelength_col], errors="coerce").to_numpy(dtype=float)
    h2o_values = np.array([parse_lut_parameter(col) for col in df.columns[1:]], dtype=float)
    spectra = df.iloc[:, 1:].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float).T
    spectra *= scale_factor

    if np.isnan(wavelengths).any() or np.isnan(spectra).any():
        raise ValueError("LUT contains NaN after numeric conversion. Check the CSV format.")

    order = np.argsort(h2o_values)
    h2o_values = h2o_values[order]
    spectra = spectra[order, :]

    lut = np.vstack([wavelengths, spectra])
    return lut, h2o_values, df


def parse_hisui_wave_column(column_name):
    match = re.fullmatch(r"wave_([-+0-9.]+)nm", str(column_name))
    if match is None:
        return None
    return float(match.group(1))


def load_hisui_spectra_csv(path, scale_factor=1.0):
    df = pd.read_csv(path)
    wave_cols = [col for col in df.columns if parse_hisui_wave_column(col) is not None]
    if not {"y", "x"}.issubset(df.columns):
        raise ValueError("HISUI spectra CSV must contain y and x columns.")
    if not wave_cols:
        raise ValueError("No wave_...nm columns were found in HISUI spectra CSV.")

    wavelengths = np.array([parse_hisui_wave_column(col) for col in wave_cols], dtype=float)
    order = np.argsort(wavelengths)
    wave_cols = [wave_cols[i] for i in order]
    wavelengths = wavelengths[order]

    values = df[wave_cols].apply(pd.to_numeric, errors="coerce") * scale_factor
    out = pd.concat([df[["y", "x"]].reset_index(drop=True), values.reset_index(drop=True)], axis=1)
    return out, wavelengths, wave_cols


def get_hisui_pixel_spectrum(df, wavelengths, pixel_y=None, pixel_x=None, row_index=0):
    if pixel_y is not None and pixel_x is not None:
        mask = (df["y"] == pixel_y) & (df["x"] == pixel_x)
        if not mask.any():
            raise ValueError(f"Pixel y={pixel_y}, x={pixel_x} was not found in HISUI spectra CSV.")
        row = df.loc[mask].iloc[0]
    else:
        row = df.iloc[int(row_index)]

    radiance = row.iloc[2:].to_numpy(dtype=float)
    spectrum = pd.DataFrame({"Wavelength": wavelengths, "Radiance": radiance})
    return spectrum, int(row["y"]), int(row["x"])


def select_hisui_bands(spectrum, mode="index", band_start=0, band_stop=None, wave_min=None, wave_max=None):
    if mode == "index":
        if band_stop is None:
            band_stop = len(spectrum)
        selected = spectrum.iloc[int(band_start):int(band_stop)].copy()
    elif mode == "wavelength":
        selected = spectrum.copy()
        if wave_min is not None:
            selected = selected[selected["Wavelength"] >= float(wave_min)]
        if wave_max is not None:
            selected = selected[selected["Wavelength"] <= float(wave_max)]
    else:
        raise ValueError('mode must be "index" or "wavelength".')

    if selected.empty:
        raise ValueError("No HISUI bands were selected.")
    return selected.reset_index(drop=True)


def select_lut_range(lut, mode="index", index_start=0, index_stop=None, wave_min=None, wave_max=None):
    wavelengths = lut[0, :]
    if mode == "index":
        if index_stop is None:
            index_stop = lut.shape[1]
        selected = lut[:, int(index_start):int(index_stop)]
    elif mode == "wavelength":
        mask = np.ones(wavelengths.shape, dtype=bool)
        if wave_min is not None:
            mask &= wavelengths >= float(wave_min)
        if wave_max is not None:
            mask &= wavelengths <= float(wave_max)
        selected = lut[:, mask]
    else:
        raise ValueError('mode must be "index" or "wavelength".')

    if selected.shape[1] < 3:
        raise ValueError("LUT selection is too small. Select at least 3 wavelength samples.")
    return selected

## 3. LM フィッティングの関数

In [ ]:
def instrumental_function(lut, sigma, mu):
    """Convolve each LUT spectrum with a Gaussian instrumental function."""
    wavelengths = lut[0, :]
    out = np.zeros_like(lut, dtype=float)
    out[0, :] = wavelengths

    center = np.mean(wavelengths)
    sigma = abs(float(sigma)) + 1e-12
    gaussian = np.exp(-((wavelengths - center - mu) ** 2) / (2 * sigma**2))
    gaussian /= gaussian.sum() + 1e-12

    for row in range(1, lut.shape[0]):
        out[row, :] = np.convolve(lut[row, :], gaussian, mode="same")
    return out


def reflectance_correction(lut, a, b, k):
    out = lut.copy()
    x = out[0, :]
    for row in range(1, out.shape[0]):
        out[row, :] = (a + b * x) * out[row, :] + k
    return out


def interpolate_lut_by_h2o(lut, h2o_values, h2o):
    """Interpolate LUT spectra along the H2O axis."""
    h2o_values = np.asarray(h2o_values, dtype=float)
    spectra = lut[1:, :]
    h2o = float(h2o)
    return np.array([np.interp(h2o, h2o_values, spectra[:, col]) for col in range(spectra.shape[1])])


def interpolate_to_hisui_wavelengths(lut_wavelengths, lut_radiance, hisui_wavelengths):
    return np.interp(hisui_wavelengths, lut_wavelengths, lut_radiance)


def model_h2o(lut_selected, h2o_values, hisui_selected, sigma, mu, a, b, h2o, k):
    out = instrumental_function(lut_selected, sigma, mu)
    out = reflectance_correction(out, a, b, k)
    lut_radiance = interpolate_lut_by_h2o(out, h2o_values, h2o)
    return interpolate_to_hisui_wavelengths(out[0, :], lut_radiance, hisui_selected["Wavelength"].to_numpy())


def residuals_h2o(params, lut_selected, h2o_values, hisui_selected):
    sigma, mu, a, b, h2o, k = params
    estimated = model_h2o(lut_selected, h2o_values, hisui_selected, sigma, mu, a, b, h2o, k)
    observed = hisui_selected["Radiance"].to_numpy(dtype=float)
    return observed - estimated


def estimate_b_from_pixel(lut_selected, h2o_values, hisui_selected, sigma=6.5, mu=0.0, h2o=None):
    if h2o is None:
        h2o = float(np.median(h2o_values))

    observed = hisui_selected["Radiance"].to_numpy(dtype=float)
    wavelengths = hisui_selected["Wavelength"].to_numpy(dtype=float)
    rel_max_index = int(np.nanargmax(observed))
    max_ref = observed[rel_max_index]
    max_wave = wavelengths[rel_max_index]

    out = instrumental_function(lut_selected, sigma=sigma, mu=mu)
    lut_radiance = interpolate_lut_by_h2o(out, h2o_values, h2o)
    model_at_hisui = interpolate_to_hisui_wavelengths(out[0, :], lut_radiance, wavelengths)
    modtran = model_at_hisui[rel_max_index]

    if not np.isfinite(modtran) or abs(max_wave * modtran) < 1e-12:
        return 1.0
    return max_ref / max_wave / modtran


def show_used_wavelength_ranges(lut_selected, hisui_selected, label="H2O"):
    w_lut = lut_selected[0, :]
    w_hisui = hisui_selected["Wavelength"].to_numpy(dtype=float)

    def stats(name, w):
        step = np.diff(w)
        print(
            f"{name}: [{w[0]:.3f} .. {w[-1]:.3f}] nm | N={w.size} | "
            f"step~{np.median(step):.3f} nm (min={step.min():.3f}, max={step.max():.3f}) | "
            f"ascending={np.all(step > 0)}"
        )

    print(f"=== {label} wavelength ranges ===")
    stats("LUT  ", w_lut)
    stats("HISUI", w_hisui)
    inside = (w_hisui.min() >= w_lut.min()) and (w_hisui.max() <= w_lut.max())
    print("Coverage:", "OK (HISUI is inside LUT)" if inside else "WARNING: HISUI range extends outside LUT")
    return w_lut, w_hisui

## 4. データ読み込みと使用バンドの確認

In [ ]:
lut, h2o_values, lut_raw_df = load_lut_csv(lut_csv, scale_factor=lut_scale_factor)

active_hisui_spectra_csv = hisui_spectra_csv

hisui_df, hisui_wavelengths, hisui_wave_cols = load_hisui_spectra_csv(active_hisui_spectra_csv, scale_factor=hisui_scale_factor)

hisui_spectrum, used_y, used_x = get_hisui_pixel_spectrum(
    hisui_df,
    hisui_wavelengths,
    pixel_y=pixel_y,
    pixel_x=pixel_x,
    row_index=hisui_row_index,
)

hisui_selected = select_hisui_bands(
    hisui_spectrum,
    mode=hisui_selection_mode,
    band_start=hisui_band_start,
    band_stop=hisui_band_stop,
    wave_min=hisui_wavelength_min_nm,
    wave_max=hisui_wavelength_max_nm,
)

lut_selected = select_lut_range(
    lut,
    mode=lut_selection_mode,
    index_start=lut_index_start,
    index_stop=lut_index_stop,
    wave_min=lut_wavelength_min_nm,
    wave_max=lut_wavelength_max_nm,
)

print(f"LUT: {lut_csv}")
print(f"H2O LUT values: {h2o_values[0]:.3f} .. {h2o_values[-1]:.3f} (N={len(h2o_values)})")
print(f"HISUI spectra CSV: {active_hisui_spectra_csv}")
print(f"Selected pixel: y={used_y}, x={used_x}")
print(f"Selected HISUI bands: N={len(hisui_selected)}")

w_lut, w_hisui = show_used_wavelength_ranges(lut_selected, hisui_selected, label="H2O")
display(hisui_selected)

## 5. LM 法でフィッティングして表示

In [ ]:
if estimate_initial_b_from_pixel:
    b0 = estimate_b_from_pixel(
        lut_selected,
        h2o_values,
        hisui_selected,
        sigma=initial_sigma,
        mu=initial_mu,
        h2o=initial_h2o,
    )
else:
    b0 = initial_b

a0 = np.array([initial_sigma, initial_mu, initial_a, b0, initial_h2o, initial_k], dtype=float)
res = least_squares(
    residuals_h2o,
    a0,
    args=(lut_selected, h2o_values, hisui_selected),
    method=least_squares_method,
    max_nfev=max_nfev,
)

print("Initial parameters [sigma, mu, a, b, h2o, k]:", a0)
print("Optimized parameters [sigma, mu, a, b, h2o, k]:", res.x)
print("Estimated H2O value:", res.x[4])
print("Success:", res.success, "|", res.message)

w_int = model_h2o(lut_selected, h2o_values, hisui_selected, *a0)
w_est = model_h2o(lut_selected, h2o_values, hisui_selected, *res.x)
w_obs = hisui_selected["Radiance"].to_numpy(dtype=float)
wavelength = hisui_selected["Wavelength"].to_numpy(dtype=float)

plt.figure(figsize=(8, 5))
plt.title(f"H2O fit: y={used_y}, x={used_x}, H2O={res.x[4]:.4f}")
plt.plot(wavelength, w_int, "-o", color="blue", label="initial", markersize=5)
plt.plot(wavelength, w_est, "-o", color="red", label="estimated", markersize=5)
plt.plot(wavelength, w_obs, "-o", color="black", label="original", markersize=5)
plt.xlabel("wavelength [nm]")
plt.ylabel("Radiance [W/m$^2$/str/nm]")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## 6. 残差の確認

In [ ]:
residual = w_obs - w_est

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(wavelength, residual, "-o", color="purple", markersize=5)
axes[0].axhline(0, color="gray", linewidth=1)
axes[0].set_title("Residual: original - estimated")
axes[0].set_xlabel("wavelength [nm]")
axes[0].set_ylabel("Radiance residual")
axes[0].grid(alpha=0.25)

axes[1].plot(w_lut, np.zeros_like(w_lut), ".", color="lightgray", label="LUT samples")
axes[1].plot(wavelength, np.zeros_like(wavelength), "o", color="black", label="HISUI bands")
axes[1].set_yticks([])
axes[1].set_title("Used wavelength samples")
axes[1].set_xlabel("wavelength [nm]")
axes[1].legend()
axes[1].grid(alpha=0.25)

plt.tight_layout()
plt.show()